[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/01_tag_1_machine_learning_playground.ipynb)

# Tag 1 Machine Learning Playground

Dieses Notebook begleitet Tag 1 praktisch. Es zeigt Rohdaten, Lossfunktionen, Regularisierung, Regression, Klassifikation, Kreuzvalidierung, Clustering, Bias-Varianz und Metrikwahl mit kleinen Datensätzen ohne Internet-Download.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from sklearn.datasets import make_regression, make_classification, load_diabetes, load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, KFold, learning_curve
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score, median_absolute_error,
    mean_absolute_percentage_error, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score, average_precision_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay, silhouette_score, adjusted_rand_score,
    normalized_mutual_info_score
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.figsize'] = (7, 4)
print('Setup abgeschlossen.')

## 2. Erst nur Rohdaten ansehen

Bevor wir über Modelle, Lossfunktionen oder Optimierung sprechen, schauen wir nur auf Daten. Der Datensatz ist synthetisch und eindimensional: `x` ist ein einzelnes Feature, `y` ist die Zielgröße. Rote Punkte sind Ausreißer. Noch wird **kein Modell** trainiert und keine Lossfunktion berechnet.

In [ ]:
def raw_data_demo(samples=80, rauschen=0.7, ausreisser=4, ausreisser_staerke=8.0):
    rng = np.random.default_rng(RANDOM_STATE)
    X_raw = np.linspace(-3, 3, samples)
    y_clean = 1.8 * X_raw - 0.5
    y_raw = y_clean + rng.normal(0, rauschen, samples)
    outlier_idx = np.array([], dtype=int)
    if ausreisser > 0:
        outlier_idx = rng.choice(samples, size=min(ausreisser, samples), replace=False)
        y_raw[outlier_idx] += ausreisser_staerke
    plt.scatter(X_raw, y_raw, alpha=0.75, label='Rohdaten')
    if len(outlier_idx):
        plt.scatter(X_raw[outlier_idx], y_raw[outlier_idx], color='red', s=80, label='Ausreißer')
    plt.xlabel('Feature x')
    plt.ylabel('Zielgröße y')
    plt.title('Rohdaten mit Rauschen und Ausreißern')
    plt.legend(); plt.show()

widgets.interact(
    raw_data_demo,
    samples=widgets.IntSlider(value=80, min=30, max=200, step=10),
    rauschen=widgets.FloatSlider(value=0.7, min=0.0, max=3.0, step=0.1),
    ausreisser=widgets.IntSlider(value=4, min=0, max=20),
    ausreisser_staerke=widgets.FloatSlider(value=8.0, min=-15.0, max=20.0, step=0.5),
);

## 3. Modell, Lossfunktion, Lernen und Regularisierung

Jetzt kommt ein Modell dazu. Du stellst manuell eine Gerade `ŷ = m · x + b` ein. Zusätzlich trainiert das Notebook eine Ridge Regression auf denselben Daten. So kannst du deine manuelle Gerade mit einem Modell vergleichen, das seine Parameter aus den Daten lernt.

- **MSE** bestraft große Fehler stark.
- **MAE** ist robuster gegen Ausreißer.
- **Huber** liegt zwischen MSE und MAE.
- **Regularisierung** bestraft große Parameter und kann Ausreißer-Effekte abmildern.

In [ ]:
def loss_values(y_true, y_pred, loss='MSE'):
    errors = y_true - y_pred
    if loss == 'MAE / L1':
        return np.mean(np.abs(errors))
    if loss == 'Huber':
        delta = 1.0
        abs_err = np.abs(errors)
        return np.mean(np.where(abs_err <= delta, 0.5 * errors**2, delta * (abs_err - 0.5 * delta)))
    return np.mean(errors**2)

def model_loss_demo(samples=80, rauschen=0.7, ausreisser=4, ausreisser_staerke=8.0, m=2.0, b=0.0, loss='MSE', regularisierung=0.0):
    rng = np.random.default_rng(RANDOM_STATE)
    X_line = np.linspace(-3, 3, samples)
    y_clean = 1.8 * X_line - 0.5
    y = y_clean + rng.normal(0, rauschen, samples)
    outlier_idx = np.array([], dtype=int)
    if ausreisser > 0:
        outlier_idx = rng.choice(samples, size=min(ausreisser, samples), replace=False)
        y[outlier_idx] += ausreisser_staerke
    X2d = X_line.reshape(-1, 1)
    manual_pred = m * X_line + b
    learned = Ridge(alpha=regularisierung, random_state=RANDOM_STATE).fit(X2d, y)
    learned_pred = learned.predict(X2d)
    manual_loss = loss_values(y, manual_pred, loss) + regularisierung * (m**2 + b**2)
    learned_loss = loss_values(y, learned_pred, loss) + regularisierung * float(np.sum(learned.coef_**2))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].scatter(X_line, y, alpha=0.75, label='Trainingsdaten')
    if len(outlier_idx):
        axes[0].scatter(X_line[outlier_idx], y[outlier_idx], color='red', s=80, label='Ausreißer')
    axes[0].plot(X_line, y_clean, 'g--', label='wahre Funktion')
    axes[0].plot(X_line, manual_pred, color='black', label=f'manuell: m={m:.1f}, b={b:.1f}')
    axes[0].plot(X_line, learned_pred, color='orange', linewidth=2, label='gelerntes Ridge-Modell')
    axes[0].set_title(f'Manueller Loss={manual_loss:.2f} | Gelernter Loss={learned_loss:.2f}')
    axes[0].set_xlabel('Feature x'); axes[0].set_ylabel('Ziel y'); axes[0].legend()
    axes[1].scatter(X_line, y - manual_pred, alpha=0.65, label='manuelle Gerade')
    axes[1].scatter(X_line, y - learned_pred, alpha=0.65, label='gelerntes Modell')
    axes[1].axhline(0, color='red', linestyle='--')
    axes[1].set_title('Residuenvergleich')
    axes[1].set_xlabel('Feature x'); axes[1].set_ylabel('Fehler')
    axes[1].legend()
    plt.tight_layout(); plt.show()

widgets.interact(
    model_loss_demo,
    samples=widgets.IntSlider(value=80, min=30, max=200, step=10),
    rauschen=widgets.FloatSlider(value=0.7, min=0.0, max=3.0, step=0.1),
    ausreisser=widgets.IntSlider(value=4, min=0, max=20),
    ausreisser_staerke=widgets.FloatSlider(value=8.0, min=-15.0, max=20.0, step=0.5),
    m=widgets.FloatSlider(value=2.0, min=-2.0, max=5.0, step=0.1),
    b=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1),
    loss=widgets.Dropdown(options=['MSE', 'MAE / L1', 'Huber'], value='MSE'),
    regularisierung=widgets.FloatSlider(value=0.0, min=0.0, max=20.0, step=0.5),
);

## 4. Regression: Daten verstehen, Splits und Kreuzvalidierung

**Diabetes-Datensatz:** `X` enthält standardisierte medizinische Features pro Person. `y` ist ein numerischer Krankheitsprogressionswert. Wir sagen also keinen Namen oder keine Klasse vorher, sondern einen kontinuierlichen Zielwert.

**Synthetische Mietpreise:** `Wohnfläche`, `Zimmer` und `Entfernung Zentrum` sind Inputs. `Miete` ist die Zielgröße. Dieser Datensatz ist nur für anschauliche Plots gedacht.

In [ ]:
diabetes = load_diabetes(as_frame=True)
X = diabetes.data
y = diabetes.target
print('Diabetes X: Input-Features')
display(X.head())
print('Diabetes y: Zielgröße Krankheitsprogression')
display(y.head().to_frame('target'))

rng = np.random.default_rng(RANDOM_STATE)
rent_df = pd.DataFrame({
    'Wohnfläche': rng.normal(75, 18, 140).clip(25, 150),
    'Zimmer': rng.integers(1, 6, 140),
    'Entfernung Zentrum': rng.uniform(0.5, 18, 140),
})
rent_df['Miete'] = 300 + 9 * rent_df['Wohnfläche'] + 120 * rent_df['Zimmer'] - 18 * rent_df['Entfernung Zentrum'] + rng.normal(0, 90, len(rent_df))
print('Synthetische Mietpreise: Inputs und Zielgröße')
display(rent_df.head())
plt.scatter(rent_df['Wohnfläche'], rent_df['Miete'], alpha=0.75)
plt.xlabel('Wohnfläche'); plt.ylabel('Miete'); plt.title('Mietpreis-Rohdaten')
plt.show()

### Train/Valid/Test Split und Kreuzvalidierung

Ein einzelner Split kann zufällig optimistisch oder pessimistisch sein. Kreuzvalidierung trainiert mehrere Splits und liefert oft eine realistischere Einschätzung des Validierungsfehlers.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_STATE)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models_reg = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=2.0, random_state=RANDOM_STATE),
    'DecisionTreeRegressor': DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=80, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}
rows = []
for name, model in models_reg.items():
    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)
    cv_mae = -cross_val_score(model, X_train_full, y_train_full, cv=cv, scoring='neg_mean_absolute_error').mean()
    rows.append({
        'Modell': name,
        'Valid MAE einzelner Split': mean_absolute_error(y_valid, valid_pred),
        'CV MAE Mittelwert': cv_mae,
        'Test RMSE': np.sqrt(mean_squared_error(y_test, test_pred)),
        'Test R2': r2_score(y_test, test_pred),
    })
display(pd.DataFrame(rows).sort_values('CV MAE Mittelwert'))

### Warum Kreuzvalidierung beim Tuning hilft

Hier optimieren wir den Polynomgrad einmal auf einem einzelnen Validierungssplit und einmal per Kreuzvalidierung. Der einzelne Split kann zufällig einen zu komplexen Grad bevorzugen. CV mittelt über mehrere Splits und ist auf Testdaten oft robuster.

In [ ]:
degrees = range(1, 13)
X_small = X[['bmi']].to_numpy()
y_small = y.to_numpy()
X_train_cv, X_test_cv, y_train_cv, y_test_cv = train_test_split(X_small, y_small, test_size=0.25, random_state=RANDOM_STATE)
X_tr_cv, X_val_cv, y_tr_cv, y_val_cv = train_test_split(X_train_cv, y_train_cv, test_size=0.30, random_state=3)
rows = []
for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1.0))
    model.fit(X_tr_cv, y_tr_cv)
    val_rmse = np.sqrt(mean_squared_error(y_val_cv, model.predict(X_val_cv)))
    cv_rmse = -cross_val_score(model, X_train_cv, y_train_cv, cv=5, scoring='neg_root_mean_squared_error').mean()
    rows.append({'degree': degree, 'Validierungssplit RMSE': val_rmse, 'CV RMSE': cv_rmse})
tuning_df = pd.DataFrame(rows)
best_split_degree = int(tuning_df.loc[tuning_df['Validierungssplit RMSE'].idxmin(), 'degree'])
best_cv_degree = int(tuning_df.loc[tuning_df['CV RMSE'].idxmin(), 'degree'])

def fit_degree(degree):
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1.0))
    model.fit(X_train_cv, y_train_cv)
    return np.sqrt(mean_squared_error(y_test_cv, model.predict(X_test_cv)))
print('Bester Grad laut einzelnem Split:', best_split_degree, '| Test RMSE:', round(fit_degree(best_split_degree), 2))
print('Bester Grad laut CV:', best_cv_degree, '| Test RMSE:', round(fit_degree(best_cv_degree), 2))
plt.plot(tuning_df['degree'], tuning_df['Validierungssplit RMSE'], marker='o', label='einzelner Validierungssplit')
plt.plot(tuning_df['degree'], tuning_df['CV RMSE'], marker='o', label='5-fold CV')
plt.xlabel('Polynomgrad')
plt.ylabel('RMSE')
plt.title('Tuning mit Split vs. Kreuzvalidierung')
plt.legend(); plt.show()

### Hyperparameter und Validierungsfehler

Dieser Block nutzt den Diabetes-Datensatz. Statt nur Modelle nebeneinander zu zeigen, wird bei jeder Änderung neu trainiert und der Fehler auf Training, Validierung und Test angezeigt. So sieht man, ob ein Hyperparameter wirklich hilft oder nur den Trainingsfehler senkt.

In [ ]:
def hyperparameter_demo(modell='Ridge', alpha=2.0, max_depth=4, n_estimators=80):
    if modell == 'Ridge':
        model = Ridge(alpha=alpha, random_state=RANDOM_STATE)
    elif modell == 'DecisionTreeRegressor':
        model = DecisionTreeRegressor(max_depth=max_depth, random_state=RANDOM_STATE)
    else:
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_valid = model.predict(X_valid)
    pred_test = model.predict(X_test)
    errors = pd.Series({
        'Train RMSE': np.sqrt(mean_squared_error(y_train, pred_train)),
        'Valid RMSE': np.sqrt(mean_squared_error(y_valid, pred_valid)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, pred_test)),
    })
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    errors.plot(kind='bar', ax=axes[0], color=['#4c78a8', '#f58518', '#54a24b'])
    axes[0].set_ylabel('RMSE')
    axes[0].set_title('Fehlervergleich')
    axes[1].scatter(y_test, pred_test, alpha=0.75, label='Testpunkte')
    axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Idealdiagonale')
    axes[1].set_xlabel('y true'); axes[1].set_ylabel('y pred')
    axes[1].set_title(f'{modell}: Test R²={r2_score(y_test, pred_test):.2f}')
    axes[1].legend()
    plt.tight_layout(); plt.show()

widgets.interact(
    hyperparameter_demo,
    modell=widgets.Dropdown(options=['Ridge', 'DecisionTreeRegressor', 'RandomForestRegressor'], value='Ridge'),
    alpha=widgets.FloatSlider(value=2.0, min=0.0, max=30.0, step=0.5),
    max_depth=widgets.IntSlider(value=4, min=1, max=12),
    n_estimators=widgets.IntSlider(value=80, min=10, max=200, step=10),
);

### Ausreißer sichtbar in Trainingsdaten und Modell

Hier nutzen wir absichtlich einen einfachen 1D-Mietpreis-Datensatz, damit die Ausreißer als rote Punkte sichtbar sind. Das Modell wird mit den Ausreißern neu trainiert.

In [ ]:
def outlier_training_demo(anzahl_ausreisser=4, staerke=450):
    rng = np.random.default_rng(RANDOM_STATE)
    flaeche = np.linspace(30, 150, 120)
    true_rent = 350 + 10 * flaeche + 0.025 * flaeche**2
    rent = true_rent + rng.normal(0, 90, len(flaeche))
    X_all = flaeche.reshape(-1, 1)
    Xtr, Xte, ytr, yte = train_test_split(X_all, rent, test_size=0.3, random_state=RANDOM_STATE)
    ytr_out = ytr.copy()
    outlier_idx = rng.choice(len(ytr_out), size=min(anzahl_ausreisser, len(ytr_out)), replace=False)
    ytr_out[outlier_idx] += staerke
    clean_model = LinearRegression().fit(Xtr, ytr)
    out_model = LinearRegression().fit(Xtr, ytr_out)
    grid = np.linspace(30, 150, 250).reshape(-1, 1)
    pred_test = out_model.predict(Xte)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].scatter(Xtr[:, 0], ytr_out, alpha=0.65, label='Training')
    axes[0].scatter(Xtr[outlier_idx, 0], ytr_out[outlier_idx], color='red', s=80, label='Ausreißer')
    axes[0].plot(grid[:, 0], 350 + 10 * grid[:, 0] + 0.025 * grid[:, 0]**2, 'g--', label='wahre Funktion')
    axes[0].plot(grid[:, 0], clean_model.predict(grid), color='gray', label='ohne Ausreißer')
    axes[0].plot(grid[:, 0], out_model.predict(grid), color='black', label='mit Ausreißern')
    axes[0].set_xlabel('Wohnfläche'); axes[0].set_ylabel('Miete'); axes[0].legend()
    axes[1].scatter(yte, pred_test, alpha=0.75, label='Testpunkte')
    axes[1].plot([yte.min(), yte.max()], [yte.min(), yte.max()], 'r--', label='Idealdiagonale')
    axes[1].set_xlabel('y true'); axes[1].set_ylabel('y pred')
    axes[1].set_title(f'Test: MAE={mean_absolute_error(yte, pred_test):.1f}, RMSE={np.sqrt(mean_squared_error(yte, pred_test)):.1f}')
    axes[1].legend()
    plt.tight_layout(); plt.show()

widgets.interact(
    outlier_training_demo,
    anzahl_ausreisser=widgets.IntSlider(value=4, min=0, max=20),
    staerke=widgets.IntSlider(value=450, min=-800, max=1200, step=50),
);

### Regressionsmetriken sichtbar machen

Die Balken zeigen, wie einzelne Fehler in MAE und RMSE eingehen. RMSE quadriert Fehler: große Fehler werden dadurch sichtbar stärker gewichtet.

In [ ]:
def regression_metric_demo(grosser_fehler=80):
    y_true_demo = np.array([100, 120, 140, 160, 180, 200])
    y_pred_demo = np.array([102, 116, 142, 158, 182, 200 + grosser_fehler])
    errors = y_pred_demo - y_true_demo
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(y_true_demo, y_pred_demo, s=80)
    axes[0].plot([y_true_demo.min(), y_true_demo.max()], [y_true_demo.min(), y_true_demo.max()], 'r--', label='Idealdiagonale')
    axes[0].set_xlabel('y true'); axes[0].set_ylabel('y pred'); axes[0].legend()
    axes[1].bar(range(len(errors)), np.abs(errors)); axes[1].set_title(f'Absolute Fehler | MAE={mean_absolute_error(y_true_demo, y_pred_demo):.1f}')
    axes[2].bar(range(len(errors)), errors**2); axes[2].set_title(f'Quadrierte Fehler | RMSE={np.sqrt(mean_squared_error(y_true_demo, y_pred_demo)):.1f}')
    plt.tight_layout(); plt.show()
widgets.interact(regression_metric_demo, grosser_fehler=widgets.IntSlider(value=80, min=0, max=220, step=10));

## 5. Klassifikation: Daten, Feature-Verteilungen und Thresholds

**Breast Cancer:** Jede Zeile beschreibt gemessene Eigenschaften eines Zellkerns. `target=0` bedeutet `malignant` (bösartig), `target=1` bedeutet `benign` (gutartig). Gerade deshalb sind Recall, Precision und Thresholds wichtig.

**Digits:** kleine 8x8-Bilder von Ziffern. Wir nutzen sie nur kurz, um Mehrklassen-Klassifikation sichtbar zu machen.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
Xc = cancer.data
yc = cancer.target
print('Breast Cancer Inputs X:')
display(Xc.head())
print('Target-Verteilung: 0=malignant, 1=benign')
display(pd.Series(yc).map(dict(enumerate(cancer.target_names))).value_counts().rename('Anzahl').to_frame())

digits = load_digits()
fig, axes = plt.subplots(1, 6, figsize=(8, 2))
for ax, image, label in zip(axes, digits.images[:6], digits.target[:6]):
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Label {label}')
    ax.axis('off')
plt.suptitle('Digits: Beispielbilder')
plt.show()

### Feature-Verteilungen und Feature-Kombinationen

Mit den Dropdowns wählst du Merkmale aus. Links sieht man die Verteilung eines Features pro Klasse. Rechts sieht man zwei Features gemeinsam. Gute Feature-Kombinationen trennen die Klassen sichtbar besser.

In [ ]:
def feature_explorer(feature_x='mean radius', feature_y='mean texture'):
    df = cancer.frame.copy()
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for target_value, name in enumerate(cancer.target_names):
        values = df.loc[df['target'] == target_value, feature_x]
        axes[0].hist(values, bins=20, alpha=0.55, label=name)
    axes[0].set_title(f'Verteilung von {feature_x}')
    axes[0].set_xlabel(feature_x); axes[0].set_ylabel('Anzahl'); axes[0].legend()

    for target_value, name in enumerate(cancer.target_names):
        part = df[df['target'] == target_value]
        axes[1].scatter(part[feature_x], part[feature_y], alpha=0.65, label=name)
    axes[1].set_xlabel(feature_x); axes[1].set_ylabel(feature_y)
    axes[1].set_title('Rohdaten: zwei Features')
    axes[1].legend()

    X_two = df[[feature_x, feature_y]].to_numpy()
    y_two = df['target'].to_numpy()
    Xtr, Xte, ytr, yte = train_test_split(X_two, y_two, test_size=0.25, stratify=y_two, random_state=RANDOM_STATE)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    clf.fit(Xtr, ytr)
    score = clf.score(Xte, yte)
    xx, yy = np.meshgrid(
        np.linspace(X_two[:, 0].min(), X_two[:, 0].max(), 120),
        np.linspace(X_two[:, 1].min(), X_two[:, 1].max(), 120),
    )
    zz = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    axes[2].contourf(xx, yy, zz, alpha=0.2, levels=2, cmap='coolwarm')
    for target_value, name in enumerate(cancer.target_names):
        part = df[df['target'] == target_value]
        axes[2].scatter(part[feature_x], part[feature_y], alpha=0.6, label=name)
    axes[2].set_title(f'Logistic Regression mit 2 Features | Test Accuracy={score:.3f}')
    axes[2].set_xlabel(feature_x); axes[2].set_ylabel(feature_y)
    plt.tight_layout(); plt.show()

widgets.interact(
    feature_explorer,
    feature_x=widgets.Dropdown(options=list(Xc.columns), value='mean radius'),
    feature_y=widgets.Dropdown(options=list(Xc.columns), value='mean texture'),
);

### Klassifikationsmodelle und Threshold-Demo

Wir trainieren auf Breast Cancer. Die Logistic Regression liefert Wahrscheinlichkeiten. Der Threshold entscheidet, ab wann wir Klasse `benign` vorhersagen. Für maximalen Recall einer Klasse muss man den Threshold bis an die Ränder bewegen können; deshalb läuft der Slider von 0.0 bis 1.0.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=RANDOM_STATE)
models_clf = {
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    'KNeighborsClassifier': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'DecisionTreeClassifier': DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    'RandomForestClassifier': RandomForestClassifier(n_estimators=80, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1),
}
rows=[]
for name, model in models_clf.items():
    model.fit(Xc_train, yc_train)
    pred = model.predict(Xc_test)
    proba_tmp = model.predict_proba(Xc_test)[:, 1]
    rows.append({'Modell': name, 'Accuracy': accuracy_score(yc_test, pred), 'Balanced Accuracy': balanced_accuracy_score(yc_test, pred),
                 'Precision': precision_score(yc_test, pred), 'Recall': recall_score(yc_test, pred), 'F1': f1_score(yc_test, pred),
                 'ROC AUC': roc_auc_score(yc_test, proba_tmp), 'PR AUC': average_precision_score(yc_test, proba_tmp), 'MCC': matthews_corrcoef(yc_test, pred)})
display(pd.DataFrame(rows).sort_values('F1', ascending=False))

logreg = models_clf['LogisticRegression']
proba = logreg.predict_proba(Xc_test)[:, 1]

def threshold_demo(threshold=0.50):
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(yc_test, pred).ravel()
    precision = precision_score(yc_test, pred, zero_division=0)
    recall = recall_score(yc_test, pred, zero_division=0)
    print(f'Precision benign: {precision:.3f} | Recall benign: {recall:.3f} | F1: {f1_score(yc_test, pred, zero_division=0):.3f}')
    print(f'False Positives: {fp} | False Negatives: {fn}')
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    ConfusionMatrixDisplay.from_predictions(yc_test, pred, display_labels=cancer.target_names, ax=axes[0], colorbar=False)
    axes[0].set_title(f'Matrix bei Threshold {threshold:.2f}')
    RocCurveDisplay.from_predictions(yc_test, proba, ax=axes[1])
    fpr = fp / max(1, fp + tn)
    tpr = tp / max(1, tp + fn)
    axes[1].scatter([fpr], [tpr], color='red', label='aktueller Threshold')
    axes[1].legend(loc='lower right')
    PrecisionRecallDisplay.from_predictions(yc_test, proba, ax=axes[2])
    axes[2].scatter([recall], [precision], color='red', label='aktueller Threshold')
    axes[2].legend(loc='lower left')
    plt.tight_layout(); plt.show()

widgets.interact(threshold_demo, threshold=widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01));

### Imbalanced Data Demo

Dieser synthetische Datensatz wird nur für eine Sache genutzt: Klassenungleichgewicht. `X` sind künstliche numerische Features, `y` ist eine seltene positive Klasse. So wird sichtbar, warum Accuracy allein gefährlich ist.

In [ ]:
def imbalanced_demo(positive_klasse_prozent=6, threshold=0.50):
    gewicht_pos = positive_klasse_prozent / 100
    Xi, yi = make_classification(n_samples=1200, n_features=8, n_informative=3, weights=[1 - gewicht_pos, gewicht_pos], flip_y=0.02, random_state=RANDOM_STATE)
    Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=RANDOM_STATE)
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    model.fit(Xi_train, yi_train)
    proba_i = model.predict_proba(Xi_test)[:, 1]
    pred_i = (proba_i >= threshold).astype(int)
    majority_pred = np.zeros_like(yi_test)
    print('Mehrheitsklasse Accuracy:', round(accuracy_score(yi_test, majority_pred), 3), '| Recall positiv:', round(recall_score(yi_test, majority_pred, zero_division=0), 3))
    print('Modell Accuracy:', round(accuracy_score(yi_test, pred_i), 3), '| Recall positiv:', round(recall_score(yi_test, pred_i, zero_division=0), 3), '| Precision positiv:', round(precision_score(yi_test, pred_i, zero_division=0), 3))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].bar(['Klasse 0', 'Klasse 1'], np.bincount(yi_test)); axes[0].set_title('Klassenverteilung')
    ConfusionMatrixDisplay.from_predictions(yi_test, pred_i, ax=axes[1], colorbar=False); axes[1].set_title('Confusion Matrix')
    RocCurveDisplay.from_predictions(yi_test, proba_i, ax=axes[2])
    tn, fp, fn, tp = confusion_matrix(yi_test, pred_i).ravel()
    axes[2].scatter([fp / max(1, fp + tn)], [tp / max(1, tp + fn)], color='red', label='aktueller Threshold')
    axes[2].legend(loc='lower right')
    plt.tight_layout(); plt.show()

widgets.interact(
    imbalanced_demo,
    positive_klasse_prozent=widgets.IntSlider(value=6, min=2, max=40, step=2),
    threshold=widgets.FloatSlider(value=0.50, min=0.0, max=1.0, step=0.01),
);

## 6. Unüberwachtes Lernen

**Wine:** chemische Messwerte verschiedener Weine. Beim Clustering werden Labels nicht genutzt; danach vergleichen wir Cluster mit echten Labels.

**Digits:** Bilddaten können ebenfalls mit PCA in 2D projiziert werden.

In [ ]:
wine = load_wine(as_frame=True)
Xw = wine.data
yw = wine.target
print('Wine Inputs X:')
display(Xw.head())
Xw_scaled = StandardScaler().fit_transform(Xw)
Xw_pca = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xw_scaled)

def kmeans_widget(clusteranzahl=3):
    labels = KMeans(n_clusters=clusteranzahl, n_init=10, random_state=RANDOM_STATE).fit_predict(Xw_scaled)
    print('Silhouette:', round(silhouette_score(Xw_scaled, labels), 3), '| ARI:', round(adjusted_rand_score(yw, labels), 3), '| NMI:', round(normalized_mutual_info_score(yw, labels), 3))
    plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=labels, cmap='tab10')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title(f'KMeans mit {clusteranzahl} Clustern')
    plt.show()
widgets.interact(kmeans_widget, clusteranzahl=widgets.IntSlider(value=3, min=2, max=8));

def dbscan_widget(eps=2.2, min_samples=5):
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(Xw_scaled)
    mask = labels != -1
    sil = silhouette_score(Xw_scaled[mask], labels[mask]) if len(set(labels[mask])) > 1 else np.nan
    print('Cluster inklusive Rauschen:', sorted(set(labels)), '| Silhouette:', sil)
    plt.scatter(Xw_pca[:, 0], Xw_pca[:, 1], c=labels, cmap='tab10')
    plt.xlabel('PC1'); plt.ylabel('PC2'); plt.title('DBSCAN')
    plt.show()
widgets.interact(dbscan_widget, eps=widgets.FloatSlider(value=2.2, min=0.5, max=5.0, step=0.1), min_samples=widgets.IntSlider(value=5, min=2, max=15));

## 7. Bias-Varianz-Tradeoff und Overfitting

Wir nutzen eine nichtlineare wahre Funktion mit kubischem Trend und lokaler Wellenstruktur. Die grüne Kurve ist die wahre Funktion, die schwarzen Kurven sind gelernte Modelle.

In [ ]:
def true_complex_function(x):
    return 0.12 * x**3 - 0.65 * x + 0.55 * np.cos(2.2 * x)

def make_nonlinear(n=80, noise=0.25):
    rng = np.random.default_rng(RANDOM_STATE)
    Xn = np.sort(rng.uniform(-3, 3, n)).reshape(-1, 1)
    yn = true_complex_function(Xn[:, 0]) + rng.normal(0, noise, n)
    return Xn, yn

def overfitting_demo(polynomgrad=14, trainingsdaten=25, rauschen=0.35):
    Xtr, ytr = make_nonlinear(trainingsdaten, rauschen)
    grid = np.linspace(-3, 3, 300).reshape(-1, 1)
    y_true = true_complex_function(grid[:, 0])
    model = make_pipeline(PolynomialFeatures(polynomgrad), LinearRegression())
    model.fit(Xtr, ytr)
    pred_grid = model.predict(grid)
    train_rmse = np.sqrt(mean_squared_error(ytr, model.predict(Xtr)))
    test_rmse = np.sqrt(mean_squared_error(y_true, pred_grid))
    plt.scatter(Xtr, ytr, label='Training', alpha=0.8)
    plt.plot(grid, y_true, 'g--', label='wahre Funktion')
    plt.plot(grid, pred_grid, color='black', label='Modell')
    ymin, ymax = np.percentile(np.r_[ytr, y_true], [1, 99])
    plt.ylim(ymin - 0.8, ymax + 0.8)
    plt.title(f'Grad {polynomgrad}: Train RMSE={train_rmse:.2f}, Test RMSE={test_rmse:.2f}')
    plt.legend(); plt.show()

widgets.interact(
    overfitting_demo,
    polynomgrad=widgets.IntSlider(value=14, min=1, max=25),
    trainingsdaten=widgets.IntSlider(value=25, min=10, max=120, step=5),
    rauschen=widgets.FloatSlider(value=0.35, min=0.0, max=1.0, step=0.05),
);

## 8. Metrikwahl und gebündelte Aufgaben

| Situation | Sinnvolle Metrik |
| --- | --- |
| Regression mit Ausreißern | MAE oder Median Absolute Error |
| Regression mit kritisch großen Fehlern | RMSE |
| Regression zur erklärten Varianz | R² |
| Klassifikation mit Klassenungleichgewicht | Balanced Accuracy, F1, PR AUC oder Recall |
| Medizinische Risikofälle | Recall der wichtigen Klasse |
| Hohe Fehlalarmkosten | Precision |

### Aufgaben

1. **Loss und Ausreißer:** Stelle im Loss-Block mehrere Ausreißer ein. Vergleiche MSE, MAE und Huber. Welche Lossfunktion reagiert am stärksten?
2. **Regularisierung:** Erhöhe die Regularisierung. Was passiert mit großen Parametern `w` und `b`?
3. **Regression:** Wähle im Modellvergleich Ridge und Random Forest. Verändere passende Hyperparameter und beobachte RMSE sowie Residuen.
4. **Kreuzvalidierung:** Vergleiche den einzelnen Validierungsfehler mit dem CV-Mittelwert. Warum ist CV meist stabiler?
5. **Klassifikation:** Suche zwei Breast-Cancer-Features, die die Klassen gut trennen. Welche Kombination funktioniert besonders gut?
6. **Threshold:** Stelle den Threshold sehr niedrig und sehr hoch. Wann bekommst du maximalen Recall, wann hohe Precision?
7. **Clustering:** Verändere KMeans-Clusteranzahl und DBSCAN-Parameter. Wann entstehen offensichtlich schlechte Cluster?
8. **Overfitting:** Erhöhe den Polynomgrad und reduziere die Trainingsdaten. Warum kann der Trainingsfehler niedrig sein, obwohl das Modell schlecht generalisiert?